In [115]:
import psycopg2
import pprint
from typing import Optional
from configparser import ConfigParser
import xml.etree.ElementTree as ET
import warnings
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
from utils.utils import check_port
import re
from bs4 import BeautifulSoup, Tag

warnings.filterwarnings("ignore")

In [116]:
config = ConfigParser()
config.read('configs.conf')

DBNAME = config['ESTAFF']['DBNAME']
HOST = config['ESTAFF']['HOST']
PORT = config['ESTAFF']['PORT']
USER = config['ESTAFF']['USER']
PASSWORD = config['ESTAFF']['PASSWORD']

In [117]:
check_port(HOST, int(PORT))

Хост 172.29.10.107. Порт 5432 ОТКРЫТ


True

In [118]:
conn = psycopg2.connect(dbname=DBNAME, user=USER, password=PASSWORD, host=HOST, port=PORT)

In [122]:
query = """
select  c.id as id,
        fullname, 
        gender_id, 
        age, 
        l.name as location_name,
        mobile_phone, 
        email, 
        desired_position_name,
        p1.name as profession_name_1,
        p2.name as profession_name_2,
        exp_years, 
        cv_summary_desc, 
        prev_educations, 
        prev_jobs, 
        last_job_position_name,
        last_job_finished, 
        skills,
        sf.data as  data,
        last_comment, 
        creation_date
from candidates c 
left join locations l on l.id = c.location_id
left join professions p1 on p1.id = c.idata_profession_id[2]
left join professions p2 on p2.id = c.idata_profession_id[1]
left join "(spxml_large_fields)" sf on sf.doc_id = c.id
where state_id in ('new', 'rr_interview:reserve', 'event_type_2:failed', 'job_offer:failed', 'reserve', 'rr_resume_review:succeeded', 'rr_interview:succeeded')
"""

In [123]:
df = pd.read_sql(query, conn)

In [124]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2264 entries, 0 to 2263
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   id                      2264 non-null   int64              
 1   fullname                2264 non-null   object             
 2   gender_id               2254 non-null   float64            
 3   age                     1912 non-null   float64            
 4   location_name           2018 non-null   object             
 5   mobile_phone            2264 non-null   object             
 6   email                   2264 non-null   object             
 7   desired_position_name   2264 non-null   object             
 8   profession_name_1       1527 non-null   object             
 9   profession_name_2       1513 non-null   object             
 10  exp_years               1654 non-null   float64            
 11  cv_summary_desc         2264 non-null   obj

In [125]:
df.head()

,id,fullname,gender_id,age,location_name,mobile_phone,email,desired_position_name,profession_name_1,profession_name_2,exp_years,cv_summary_desc,prev_educations,prev_jobs,last_job_position_name,last_job_finished,skills,data,last_comment,creation_date
0,7480891603610853409,Чухлей Александр,0.0,NaN,None,+375029696261,chuhlei.sasha@yandex.ru,,None,None,NaN,,None,None,,False,None,"[b'\xc2', b'\xa0', b' ', b' ', b' ', b' ', b' ...",Резерв на КМ крупный бизнес,2025-03-18 14:44:45+00:00
1,7480891603610853409,Чухлей Александр,0.0,NaN,None,+375029696261,chuhlei.sasha@yandex.ru,,None,None,NaN,,None,None,,False,None,"[b'%', b'P', b'D', b'F', b'-', b'1', b'.', b'4...",Резерв на КМ крупный бизнес,2025-03-18 14:44:45+00:00
2,7480891603610853413,Сафронова Татьяна,1.0,25.0,None,+375333539100,tanya.polezhaeva.1999@mail.ru,,None,None,NaN,,None,None,,False,None,"[b'<', b'h', b't', b'm', b'l', b' ', b'x', b'-...",Ререзв на вакансию в крупный бизнес,2025-03-18 14:49:51+00:00
3,7483473712665932760,Киклевич Елизавета Николаевна,1.0,24.0,Минск,+375291350813,liza.vorontsova.99@mail.ru,Специалист,None,None,5.0,,<prev_educations><prev_education><end_year>202...,<prev_jobs><prev_job><start_date>2019-06-01</s...,Специалист по ОРБУ 1 категории,True,<skills><skill><type_id>english</type_id><leve...,"[b'\xff', b'\xd8', b'\xff', b'\xe0', b'\x00', ...",,2025-03-19 11:15:00+00:00
4,7483473712665932760,Киклевич Елизавета Николаевна,1.0,24.0,Минск,+375291350813,liza.vorontsova.99@mail.ru,Специалист,None,None,5.0,,<prev_educations><prev_education><end_year>202...,<prev_jobs><prev_job><start_date>2019-06-01</s...,Специалист по ОРБУ 1 категории,True,<skills><skill><type_id>english</type_id><leve...,"[b'<', b'h', b't', b'm', b'l', b'>', b'\r', b'...",,2025-03-19 11:15:00+00:00


In [126]:
def extract_html_resume(x):
    if x is None:
        return None

    if isinstance(x, (bytes, bytearray, memoryview)):
        try:
            s = bytes(x).decode('utf-8', errors='ignore').strip()
        except Exception:
            return None
    else:
        s = str(x).strip()

    if "<html>" in s and '<p class="EStaffResumeTitle">' in s:
        return s
    return None

In [127]:
df['html'] = df['data'].apply(extract_html_resume)


df = df[df['html'].notna()].reset_index(drop=True).copy()

In [129]:
def get_element_text(element: ET.Element, tag: str) -> Optional[str]:
    """Универсальная функция для извлечения текста из XML элемента"""
    node = element.find(tag)
    if node is None or not node.text:
        return None
    return node.text.strip()

def parse_education_xml(xml_text: str) -> str:
    """Парсинг образования из XML"""
    if not xml_text or not isinstance(xml_text, str):
        return ""
    
    try:
        root = ET.fromstring(xml_text)
        specialities = [
            spec for education in root.findall('prev_education')
            if (spec := get_element_text(education, 'speciality_name'))
        ]
        return ', '.join(specialities)
    except ET.ParseError:
        return ""

def parse_skills_xml(xml_text: str) -> str:
    """Парсинг навыков из XML, с явным указанием уровня языка"""
    if not xml_text or not isinstance(xml_text, str):
        return ""
    lang_levels = {
        "1": "начальный",
        "2": "средний",
        "3": "уверенный",
        "4": "свободный",
    }
    try:
        root = ET.fromstring(xml_text)
        skills = []
        
        for sk in root.findall('skill'):
            type_id = get_element_text(sk, 'type_id')
            if not type_id:
                continue
            
            level_id = get_element_text(sk, 'level_id')
            level_name = lang_levels.get(level_id)
            
            if level_name:
                skills.append(f"{type_id} — {level_name}")
            elif level_id:
                skills.append(f"{type_id} — уровень {level_id}")
            else:
                skills.append(type_id)
        
        return ', '.join(skills)
    except ET.ParseError:
        return ""

def plural_years(n):
    """Склонение слова 'год'"""
    if pd.isna(n):
        return ""
    n = int(n)
    if n % 10 == 1 and n % 100 != 11:
        return "год"
    elif n % 10 in [2, 3, 4] and n % 100 not in [12, 13, 14]:
        return "года"
    return "лет"

def plural_months(n) -> str:
    """Склонение слова 'месяц'"""
    if pd.isna(n):
        return ""
    n = int(n)
    if n == 1:
        return "месяц"
    elif n in [2, 3, 4]:
        return "месяца"
    return "месяцев"

def calculate_duration(start_date_text: str, end_date_text: Optional[str]) -> str:
    """Вычисление длительности работы"""
    if not start_date_text:
        return "неизвестно"
    try:
        start = datetime.strptime(start_date_text, "%Y-%m-%d")
        if end_date_text:
            end = datetime.strptime(end_date_text, "%Y-%m-%d")
        else:
            end = datetime.today()

        if end < start:
            return "неизвестно"

        d = relativedelta(end, start)
        parts = []
        if d.years:
            parts.append(f"{d.years} {plural_years(d.years)}")
        if d.months:
            parts.append(f"{d.months} {plural_months(d.months)}")
        return " ".join(parts) if parts else "менее месяца"
    except (ValueError, TypeError):
        return "неизвестно"   

def clean_description(text: str) -> str:
    """Очистка и форматирование описания"""
    if not text:
        return "не указано"
    
    text = text.replace('*', '')
    text = text.replace('\n\n', '\n')
    text = re.sub(r'\n[-•·]\s*', '. ', text)
    text = text.replace('\n- ', '. ')
    text = text.replace('\n-', '. ')
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s*;\s*', '; ', text)
    text = re.sub(r'\.\s*;\s*', '. ', text)
    text = re.sub(r'^\s*;\s*', '', text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r';\s*\.', '.', text)
    text = re.sub(r'\s+([,.])', r'\1', text)
    text = re.sub(r'([,.:])([^\s\d])', r'\1 \2', text)
    text = text.strip()
    text = re.sub(r'[;,]\s*$', '', text)
    
    return text


def parse_jobs_xml(xml_text: str) -> str:
    """Парсинг истории работы из XML"""
    if not xml_text or not isinstance(xml_text, str):
        return ""
    
    try:
        root = ET.fromstring(xml_text)
        jobs = []

        for job in root.findall('prev_job'):
            position_name = get_element_text(job, 'position_name') or 'неизвестно'
            comment = get_element_text(job, 'comment')
            start_date = get_element_text(job, 'start_date')
            end_date = get_element_text(job, 'end_date')

            duration = calculate_duration(start_date, end_date)
            description = clean_description(comment)

            jobs.append(
                f"Должность: {position_name}. "
                f"Длительность: {duration}. "
                f"Описание: {description}."
            )

        return "\n".join(f"{i+1}. {text}" for i, text in enumerate(jobs))
    except ET.ParseError:
        return ""

def build_resume_row(row: pd.Series) -> str:
    """Формирование структурированного текста резюме для LLM из строки данных."""

    def safe_str(x) -> str:
        return "" if pd.isna(x) else str(x).strip()

    def norm_line(text: str) -> str:
        text = re.sub(r"\s+", " ", text)
        text = text.replace("..", ".")
        text = re.sub(r"\s*;\s*", "; ", text)
        text = re.sub(r"\.\s*;\s*", ". ", text)
        return text.strip()

    def norm_multiline(text: str) -> str:
        lines = text.split("\n")
        lines = [norm_line(line) for line in lines]
        while lines and lines[0] == "":
            lines.pop(0)
        while lines and lines[-1] == "":
            lines.pop()
        return "\n".join(lines)

    id = safe_str(row.get("id"))
    desired_position_name = safe_str(row.get("desired_position_name"))
    profession_name_1 = safe_str(row.get("profession_name_1"))
    profession_name_2 = safe_str(row.get("profession_name_2"))
    exp_years = row.get("exp_years")
    cv_summary_desc = safe_str(row.get("cv_summary_desc"))
    parsed_educations = safe_str(row.get("parsed_educations"))
    parsed_jobs = safe_str(row.get("parsed_jobs"))
    skills = safe_str(row.get("parsed_skills"))

    if pd.isna(exp_years):
        exp_text = "не указано"
    else:
        exp_years_int = int(exp_years)
        exp_text = f"{exp_years_int} {plural_years(exp_years_int)}"

    if profession_name_1:
        if profession_name_2:
            profession_full = f"{profession_name_1}, {profession_name_2}"
        else:
            profession_full = profession_name_1
    else:
        profession_full = "не указано"

    if cv_summary_desc:
        about_text = clean_description(cv_summary_desc)
    else:
        about_text = "не указано"

    education_text = parsed_educations if parsed_educations else "не указано"

    has_jobs = bool(parsed_jobs)

    skills_text = skills if skills else ""

    parts = []

    parts.append("=== ОБЩАЯ ИНФОРМАЦИЯ ===")
    parts.append(f"ID кандидата: {id}")
    parts.append(f"Желаемая должность: {desired_position_name or 'не указано'}")
    parts.append(f"Общее название профессии: {profession_full}")
    parts.append(f"Общий опыт работы: {exp_text}")

    parts.append("\n=== О СЕБЕ ===")
    parts.append(f"Описание кандидата: {about_text}")

    parts.append("\n=== ОБРАЗОВАНИЕ ===")
    parts.append(f"Образование описание: {education_text}")

    parts.append("\n=== ОПЫТ РАБОТЫ ===")
    if has_jobs:
        parts.append(parsed_jobs)
    else:
        parts.append("Опыт работы по местам: не указано")

    parts.append("\n=== ЯЗЫКИ ===")
    if skills_text:
        parts.append(f"Список языков: {skills_text}")
    else:
        parts.append("Список языков: не указано")


    text = "\n".join(parts)

    text = norm_multiline(text)

    return text

In [130]:
df['parsed_educations'] = df['prev_educations'].apply(parse_education_xml)
df['parsed_jobs'] = df['prev_jobs'].apply(parse_jobs_xml)
df['parsed_skills'] = df['skills'].apply(parse_skills_xml)
df["resume"] = df.apply(build_resume_row, axis=1)

In [131]:
def clean_description(text: str) -> str:
    """Очистка и форматирование описания."""
    if not text:
        return "не указано"

    text = text.replace('*', '')
    text = text.replace('\n\n', '\n')
    text = re.sub(r'\n[-•·]\s*', '. ', text)
    text = text.replace('\n- ', '. ')
    text = text.replace('\n-', '. ')
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s*;\s*', '; ', text)
    text = re.sub(r'\.\s*;\s*', '. ', text)
    text = re.sub(r'^\s*;\s*', '', text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r';\s*\.', '.', text)
    text = re.sub(r'\s+([,.])', r'\1', text)
    text = re.sub(r'([,.:])([^\s\d])', r'\1 \2', text)
    text = text.strip()
    text = re.sub(r'[;,]\s*$', '', text)
    return text


def safe_str(x) -> str:
    return "" if pd.isna(x) else str(x).strip()


def norm_line(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = text.replace("..", ".")
    text = re.sub(r"\s*;\s*", "; ", text)
    text = re.sub(r"\.\s*;\s*", ". ", text)
    return text.strip()


def norm_multiline(text: str) -> str:
    lines = text.split("\n")
    lines = [norm_line(line) for line in lines]
    while lines and lines[0] == "":
        lines.pop(0)
    while lines and lines[-1] == "":
        lines.pop()
    return "\n".join(lines)


def parse_common_info_from_html(soup: BeautifulSoup) -> dict:
    """
    Парсинг общей информация из html.
    """
    desired_position = ""
    total_exp = "не указано"
    education_short = "не указано"

    title_tag = soup.find("p", class_="EStaffResumeTitle")
    if title_tag:
        desired_position = title_tag.get_text(strip=True)

    p_tags = soup.find_all("p")
    for p in p_tags:
        text = p.get_text(" ", strip=True)
        if "Образование:" in text and "Общий стаж:" in text:
            m_edu = re.search(r"Образование:\s*(.+?)\s+Общий стаж:", text)
            if m_edu:
                education_short = m_edu.group(1).strip()

            m_exp = re.search(r"Общий стаж:\s*(.+)$", text)
            if m_exp:
                total_exp = m_exp.group(1).strip()
            break

    return {
        "desired_position": desired_position or "не указано",
        "total_experience": total_exp or "не указано",
        "education_short": education_short or "не указано",
    }


def parse_experience_from_html(soup: BeautifulSoup) -> str:
    """
    Парсинг раздела 'Опыт работы'.
    """
    exp_h2 = None
    for h2 in soup.find_all("h2", class_="EStaffResumeSectionTitle"):
        if "Опыт работы" in h2.get_text():
            exp_h2 = h2
            break

    if not exp_h2:
        return "Опыт работы по местам: не указано"

    exp_table = exp_h2.find_next("table")
    if not exp_table:
        return "Опыт работы по местам: не указано"

    rows = exp_table.find_all("tr", recursive=False)
    items = []
    index = 1

    for tr in rows:
        tds = tr.find_all("td", recursive=False)
        if len(tds) < 2:
            continue

        period_td = tds[0]
        right_td = tds[1]

        length_p = period_td.find("p", class_="EStaffResumePeriodLengthDesc")
        length_text = ""
        if length_p:
            length_text = length_p.get_text(" ", strip=True)

        pos_p = right_td.find("p", class_="EStaffResumePrevJobPositionName")
        comment_p = right_td.find("p", class_="EStaffResumePrevJobPositionComment")

        pos_name = pos_p.get_text(" ", strip=True) if pos_p else ""
        comment_raw = comment_p.decode_contents() if comment_p else ""
        comment_text = BeautifulSoup(comment_raw, "html.parser").get_text("\n")
        comment_text = clean_description(comment_text)

        header = f"{index}. Должность: {pos_name or 'не указано'}"
        if length_text:
            header += f". Длительность: {length_text}"
        else:
            header += ". Длительность: не указано"

        if comment_text and comment_text != "не указано":
            descr = f"Описание: {comment_text}"
        else:
            descr = "Описание: не указано"

        items.append(f"{header}. {descr}")
        index += 1

    if not items:
        return "Опыт работы по местам: не указано"

    return "\n".join(items)


def parse_skills_from_html(soup: BeautifulSoup) -> str:
    """Парсинг раздела 'Ключевые навыки'"""
    skills_h2 = None
    for h2 in soup.find_all("h2", class_="EStaffResumeSectionTitle"):
        if "Ключевые навыки" in h2.get_text():
            skills_h2 = h2
            break

    if not skills_h2:
        return "не указано"

    p = skills_h2.find_next("p")
    if not p:
        return "не указано"

    spans = p.find_all("span")
    skills = [s.get_text(" ", strip=True) for s in spans if s.get_text(strip=True)]
    if not skills:
        return "не указано"

    return ", ".join(skills)


def parse_about_from_html(soup: BeautifulSoup) -> str:
    """
    Парсинг раздела 'Обо мне'.
    """
    about_h2 = None
    for h2 in soup.find_all("h2", class_="EStaffResumeSectionTitle"):
        if "Обо мне" in h2.get_text():
            about_h2 = h2
            break

    if not about_h2:
        return "не указано"

    texts = []

    for sibling in about_h2.find_all_next():
        if sibling.name == "h2" and sibling is not about_h2:
            break

        if sibling.name == "p":
            only_link = (
                len(list(sibling.children)) == 1
                and isinstance(next(iter(sibling.children), None), Tag)
                and next(iter(sibling.children)).name == "a"
            )
            if only_link:
                continue
            t = sibling.get_text("\n", strip=True)
            if t:
                texts.append(t)

    if not texts:
        return "не указано"

    about_raw = "\n".join(texts)
    return clean_description(about_raw)


def parse_education_core_from_html(soup: BeautifulSoup) -> list:
    """
    Парсинг Основного образования.
    """
    edu_h2 = None
    for h2 in soup.find_all("h2", class_="EStaffResumeSectionTitle"):
        if h2.get_text(strip=True).startswith("Образование"):
            edu_h2 = h2
            break

    if not edu_h2:
        return []

    table = edu_h2.find_next("table")
    if not table:
        return []

    rows = table.find_all("tr", recursive=False)
    items = []
    for tr in rows:
        tds = tr.find_all("td", recursive=False)
        if len(tds) < 2:
            continue

        right_td = tds[1]
        p_list = right_td.find_all("p")
        if len(p_list) > 1:
            degree = p_list[1].get_text(" ", strip=True)
            if degree:
                items.append(degree)
    return items


def parse_additional_education_specialties_from_html(soup: BeautifulSoup) -> list:
    """
    Парсинг Дополнительного образования.
    """
    add_h2 = None
    for h2 in soup.find_all("h2", class_="EStaffResumeSectionTitle"):
        if "Дополнительное образование" in h2.get_text():
            add_h2 = h2
            break

    if not add_h2:
        return []

    table = add_h2.find_next("table")
    if not table:
        return []

    rows = table.find_all("tr", recursive=False)
    items = []
    for tr in rows:
        tds = tr.find_all("td", recursive=False)
        if len(tds) < 2:
            continue

        right_td = tds[1]
        p_list = right_td.find_all("p")
        if len(p_list) > 1:
            spec = p_list[1].get_text(" ", strip=True)
            if spec:
                items.append(spec)

    return items


def parse_languages_from_html(soup: BeautifulSoup) -> str:
    """
    Парсинг Знания языков.
    """
    lang_h2 = None
    for h2 in soup.find_all("h2", class_="EStaffResumeSectionTitle"):
        if "Знание языков" in h2.get_text():
            lang_h2 = h2
            break

    if not lang_h2:
        return "не указано"

    langs = []
    p = lang_h2.find_next("p")
    while p:
        text = p.get_text(" ", strip=True)
        if not text:
            nxt = p.find_next_sibling()
            if not nxt or nxt.name == "h2":
                break
            p = nxt if nxt.name == "p" else p.find_next_sibling("p")
            continue

        lang_only = "".join(
            t for t in p.find_all(string=True, recursive=False)
        ).strip()
        span_level = p.find("span", class_="EStaffResumeLanguageLevel")
        level = span_level.get_text(" ", strip=True) if span_level else ""

        if lang_only:
            if level:
                langs.append(f"{lang_only} {level}")
            else:
                langs.append(lang_only)

        nxt = p.find_next_sibling()
        if not nxt or (nxt.name == "h2" and "SectionTitle" in " ".join(nxt.get("class", []))):
            break
        p = nxt if nxt.name == "p" else p.find_next_sibling("p")

    if not langs:
        return "не указано"

    return ", ".join(langs)


def build_resume_row(row: pd.Series) -> str:
    """
    Формирование структурированного текста резюме для LLM.
    """
    id_ = safe_str(row.get("id"))
    html = row.get("html")

    profession_name_1 = safe_str(row.get("profession_name_1"))
    profession_name_2 = safe_str(row.get("profession_name_2"))
    if profession_name_1:
        if profession_name_2:
            profession_full = f"{profession_name_1}, {profession_name_2}"
        else:
            profession_full = profession_name_1
    else:
        profession_full = "не указано"

    if not html or pd.isna(html):
        return (
            "=== ОБЩАЯ ИНФОРМАЦИЯ ===\n"
            f"ID кандидата: {id_}\n"
            "Желаемая должность: не указано\n"
            f"Общее название профессии: {profession_full}\n"
            "Общий опыт работы: не указано"
        )

    soup = BeautifulSoup(str(html), "html.parser")

    common = parse_common_info_from_html(soup)
    desired_position = common["desired_position"]
    total_experience = common["total_experience"]
    education_short = common["education_short"]

    about_text = parse_about_from_html(soup)
    experience_text = parse_experience_from_html(soup)
    skills_text = parse_skills_from_html(soup)
    languages_text = parse_languages_from_html(soup)
    edu_core_list = parse_education_core_from_html(soup)
    add_specs_list = parse_additional_education_specialties_from_html(soup)

    edu_parts = []

    if education_short and education_short != "не указано":
        edu_parts.append(education_short)

    edu_parts.extend(edu_core_list)
    edu_parts.extend(add_specs_list)

    seen = set()
    edu_unique = []
    for item in edu_parts:
        item_norm = item.strip()
        if item_norm and item_norm not in seen:
            seen.add(item_norm)
            edu_unique.append(item_norm)

    if edu_unique:
        education_final = ", ".join(edu_unique)
    else:
        education_final = "не указано"

    parts = []

    parts.append("=== ОБЩАЯ ИНФОРМАЦИЯ ===")
    parts.append(f"ID кандидата: {id_}")
    parts.append(f"Желаемая должность: {desired_position or 'не указано'}")
    parts.append(f"Общее название профессии: {profession_full}")
    parts.append(f"Общий опыт работы: {total_experience or 'не указано'}")

    parts.append("\n=== О СЕБЕ ===")
    parts.append(f"Описание кандидата: {about_text or 'не указано'}")

    parts.append("\n=== ОБРАЗОВАНИЕ ===")
    parts.append(f"Образование описание: {education_final}")

    parts.append("\n=== ОПЫТ РАБОТЫ ===")
    parts.append(experience_text)

    parts.append("\n=== КЛЮЧЕВЫЕ НАВЫКИ ===")
    parts.append(f"Список ключевых навыков: {skills_text or 'не указано'}")

    parts.append("\n=== ЯЗЫКИ ===")
    parts.append(f"Список языков: {languages_text or 'не указано'}")

    text = "\n".join(parts)
    text = norm_multiline(text)
    return text

In [132]:
df["html_resume"] = df.apply(build_resume_row, axis=1)

In [113]:
print(df["html_resume"][0])

=== ОБЩАЯ ИНФОРМАЦИЯ ===
ID кандидата: 7547637147079427606
Желаемая должность: Data scientist/ML Engineer
Общее название профессии: Информационные технологии, Data Scientist
Общий опыт работы: 8 лет 8 месяцев

=== О СЕБЕ ===
Описание кандидата: Data Scientist | Machine Learning Engineer Контакты: Тел. : +357293574497 E-mail: denis_liv@mail. ru Специалист в анализе данных и машинном обучении с научным бэкграундом в экспериментальной медицине. Совмещаю глубокую исследовательскую экспертизу с практическими навыками программирования и разработки ML-решений.

=== ОБРАЗОВАНИЕ ===
Образование описание: Кандидат наук, Исследователь, Python разработчик

=== ОПЫТ РАБОТЫ ===
1. Должность: Data Scientist. Длительность: 8 месяцев. Описание: - Полный цикл процесса сбора, очистки и подготовки данных для обеспечения качества датасетов для обучения моделей. Проведение feature engineering для улучшения эффективности алгоритмов машинного обучения. Разработка, валидация и внедрение скоринговых моделей кре

In [ ]:
system_prompt = """
Ты — ИИ-помощник, который преобразует неструктурированное текстовое описание кандидата (резюме, выгрузка из ATS и т.п.) в структурированный JSON для записи в векторную БД c дальнейшей фильтрацией 
при семантическом поиске по описаниям вакансий.

Твоя задача — по тексту резюме:
1. Выделить и нормализовать ключевые метаданные кандидата.
2. Сформировать очищенный и унифицированный текст-представление кандидата для эмбеддинг-модели (для RAG-поиска по вакансиям).
3. Вернуть результат строго в формате JSON.

Требования к результату:
Формат ответа
Отвечай строго одним JSON-объектом без пояснений, без текста до или после. Никакого Markdown.
Структура JSON (фиксированная):
{
  "candidate_id": "string",
  "positions": ["string"],
  "grade": "string | null",
  "experience_years": number | null,
  "hard_skills": ["string"],
  "domain_skills": ["string"],
  "languages": [
    {
      "name": "string",
      "level": "string"
    }
  ],
  "embedding_text": "string"
}
Семантика полей
candidate_id  Строка с ID кандидата.  

positions (список подходящих позиций)  Нормализованный список релевантных названий позиций на русском, по которым этого кандидата можно искать.  
Используй:  желаемую должность;  
должности из опыта работы;  
при необходимости — краткую нормализацию/объединение (например, "Python разработчик", "Python-разработчик", "Backend-разработчик" → "Python backend разработчик").
Исключай совсем нерелевантные, случайные должности. Слишком усзоспециализированные должности стоит привести к более общей.

grade  Грейд: "Junior", "Middle", "Senior", "Lead", "Head", "Intern" и т.п.  
Определи по:  явным указаниям в должности или описании ("Senior", "Middle", "Junior") или по релевантному опыту работы, применяемым технологиям и выполняемым задачам;  
Для не IT специальностей поставь null.

experience_years  Общее количество лет релевантного профессионального опыта. 
В первую очередь используй явно указанное поле (например, "Общий опыт работы: 5 лет").  
Если общего опыта нет, аккуратно оцени по длительности позиций (годы + месяцы, округляя до десятых, например 5.5).  
Если оценка невозможна — null.  

hard_skills (ключевые технические навыки)  Список технических/профессиональных навыков, нормализованных и без дублей:  языки программирования (Python, Java, C++ и т.п.);  
фреймворки и библиотеки (Django, FastAPI и т.д.);  
инструменты и технологии (Docker, Kafka, PostgreSQL, CI/CD, GitLab, RAG, LLM и т.п.);  
бизнес-инструменты (1C, CRM-системы, кассовые аппараты, системы учета, кадровое делопроизводство и т.п.).

Строки должны быть краткими и однозначными, например:
["Python", "FastAPI", "Django", "RAG", "LLM", "OpenAI API", "PostgreSQL", "Kafka", "Docker", "GitLab", "Celery", "Pytest", "CI/CD"]

domain_skills (доменные навыки / сфера деятельности)  Это не инструменты, а предметные области и типы задач:  "разработка backend-сервисов", "проектирование микросервисной архитектуры",  
"расчетно-кассовое обслуживание", "валютно-обменные операции",  
"кадровое делопроизводство", "работа с корпоративными клиентами",  
"обслуживание физических лиц", "розничные продажи", "управление персоналом" и т.п.

Формулируй как короткие фразы:
["разработка backend-сервисов", "проектирование архитектуры", "построение RAG-систем", "расчетно-кассовое обслуживание корпоративных клиентов"]

languages (языки)  Список объектов:
{
  "name": "english",
  "level": "upper-intermediate"
}
Нормализуй уровни, если возможно, к типичным меткам:  "beginner", "elementary", "pre-intermediate", "intermediate", "upper-intermediate", "advanced", "native".

Если уровень указан описательно (например, "уверенный"), то:  "уверенный" → "upper-intermediate" или "advanced" (выбери наиболее разумно, но последовательно);  
"свободный" → "advanced" или "native" (если явно родной — "native");  
если уровень не указан → "unknown".

name — в нижнем регистре (english, german, belarusian и т.п.).

embedding_text (подготовленный текст для эмбеддинга)  Это один большой нормализованный текст для подачи в эмбеддинг-модель.  
Требования:язык: русский (переводи ключевые англоязычные термины только если это не навредит поиску, названия технологий и стеков оставляй на английском: Python, Django, Docker и т.п.);  
без лишнего шума (контакты, никнеймы, "телеграм для связи", адреса, телефоны — удалить);  
структурированность на уровне смысла, но без спец. разметки (никаких заголовков "=== ОПЫТ РАБОТЫ ===");  
включи в текст:  желаемую должность;  
основные должности в опыте;  
суммарный опыт (если есть);  
ключевые hard skills (особенно стек технологий);  
доменные навыки;  
краткое описание того, что кандидат делал (типы задач и ответственности).

Текст должен быть естественным, но плотным по смыслу, без воды.  

Общие правила обработки
Работай только с данными, явно или неявно присутствующими в тексте резюме.  
Не добавляй навыки или позиции, которых нет даже косвенно.  
Можно аккуратно обобщать и нормализовать (например, "Специалист по операционно-кассовой работе" → "специалист по операционно-кассовой работе в банке").  
Если какое-то поле невозможно определить — ставь null (для строк) или пустой список (для массивов), но всегда соблюдай структуру JSON.  
Не изменяй названия ключей в JSON и не добавляй новые.
"""

user_prompt = """
Преобразуй следующий текст резюме кандидата в JSON по описанным выше правилам:
{text}
"""

In [ ]:
system_prompt = """ 
Ты — эксперт в области HR. Твоя задача — преобразовать неструктурированное текстовое описание кандидата в строго форматированный JSON для векторной базы данных.

МЕТОД РАБОТЫ (Chain of Thought):
ШАГ 1 — АНАЛИЗ ИСХОДНОГО ТЕКСТА:
    Внимательно прочитай весь текст резюме.
    Выдели ключевые секции: личная информация, опыт работы, образование, навыки, языки.
    Определи явные и неявные указания на метаданные.
ШАГ 2 — ИЗВЛЕЧЕНИЕ МЕТАДАННЫХ:
    candidate_id: найди идентификатор.
    positions: выдели все релевантные должности, которые могут быть предложены кандидату. Нормализуй названия.
    grade: определи уровень по явным указаниям или косвенным признакам.
    experience_years: рассчитай общий релевантный опыт.
    hard_skills: извлеки релевантные технические навыки и инструменты.
    domain_skills: определи релевантные предметные области деятельности.
    languages: извлеки языки и нормализуй уровни владения.
ШАГ 3 — ФОРМИРОВАНИЕ EMBEDDING_TEXT:
    Создай плотный информативный текст на русском языке.
    Включи: желаемую позицию, релевантный ключевой опыт, основные навыки, типы задач.
    Удали личную информацию и контактные данные.
    Сохрани технические термины на английском.

САМОПРОВЕРКА (Self-Checking):
ПРОВЕРКА 1 — ПОЛНОТА ДАННЫХ:
    Все ли обязательные поля заполнены?
    Соответствуют ли данные исходному тексту?
    Нет ли пропущенных важных навыков или опыта?
ПРОВЕРКА 2 — КОРРЕКТНОСТЬ ФОРМАТА:
    Соблюдена ли структура JSON?
    Правильные типы данных для каждого поля?
    Нет ли лишних полей или измененных ключей?
ПРОВЕРКА 3 — КАЧЕСТВО ДАННЫХ:
    Нормализованы ли навыки и должности?
    Корректно ли оценен опыт и грейд?
    Естественен ли embedding_text для поиска?
ПРОВЕРКА 4 — СООТВЕТСТВИЕ ТРЕБОВАНИЯМ:
    Использованы только данные из резюме?
    Удалены ли контакты и личная информация?
    Сохранена ли семантика всех полей?
ТРЕБОВАНИЯ К ВЫВОДУ:
Отвечай ТОЛЬКО одним JSON-объектом без каких-либо пояснений, текста до или после. Не используй Markdown разметку.
Структура JSON (фиксированная):
{
  "candidate_id": "string",
  "positions": ["string"],
  "grade": "string | null",
  "experience_years": number | null,
  "hard_skills": ["string"],
  "domain_skills": ["string"],
  "languages": [
    {
      "name": "string",
      "level": "string"
    }
  ],
  "embedding_text": "string"
}
СЕМАНТИКА ПОЛЕЙ:
    candidate_id: уникальный идентификатор кандидата.
    positions: нормализованный список релевантных должностей на русском.
    grade: "Junior", "Middle", "Senior", "Lead", "Head", "Intern" или null.
    experience_years: общий релевантный опыт в годах (округляй до десятых).
    hard_skills: технические навыки и инструменты без дублей.
    domain_skills: предметные области и типы задач.
    languages: языки с нормализованными уровнями владения.
    embedding_text: очищенный структурированный текст для эмбеддинг-модели.
ВАЖНО: Работай только с данными из исходного текста. Не добавляй информацию, которой нет в резюме.
"""

user_prompt = """
Преобразуй следующий текст резюме кандидата в JSON по описанным выше правилам:
{text}
"""